In [1]:
import sys
sys.path.insert(1, 'OneDrive/Documents/GitHub/PyParse')
import pandas as pd
import math
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Descriptors
from rdkit.Chem import Draw
from rdkit.Chem import PandasTools
from statistics import mean
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns

In [2]:
def getMSData(spectrum):
    """
    Takes the specific region of the rpt file pertaining to 
    m/z data for a specific peak in a specific well, and 
    returns a 2-D list containing all m/z peaks and their 
    normalised intensity.
    
    :param spectrum: Section of rpt file as string
    
    :return: 2-D list in following format: 
        [m/z value, normalised intensity of that value]
    """
    
    masses = []
    total = 0
    
    lineData = spectrum.split(";Mass\t% BPI")[1].split("\n")
    for line in lineData[1:]:
        if line == "}": #stop the for loop if end of MSData section is reached
            break
    
        massData = line.split("\t")
        if len(massData) == 2:
            floatData = [float(i) for i in massData] #convert all data to float
            masses.append(floatData)
            
            total = total + floatData[1]
    #Remove any masses which, as a percentage, round to 0 
    #to remove unnecessary baseline ions
    refined_masses = []
    for i in masses:
        if math.floor((i[1]/total)*100) > 0:
            refined_masses.append([i[0], i[1]])
    
    return refined_masses
   
def getUVData(spectrum, min_uv_threshold):
    """
    Takes the specific region of the rpt file pertaining to 
    UV absorbance spectrum data for a specific peak in a specific
    well, and returns a list containing all the maxima of that spectrum.
    The height of the maxima must be greater than the min_uv_threshold
    specified in options.
    
    :param spectrum: Section of rpt file as string
    
    :return: UV maxima as a list
    """

    UVmaxima = []
    
    lineData = spectrum.split(";Mass\t% BPI")[1].split("\n")
    UVx = []
    UVy = []
    for line in lineData:
        
        if line == "}":#stop for loop if end of UVData section is reached
            break
        
        UVdata = line.split("\t")
        if len(UVdata) == 2:
            UVx.append(float(UVdata[0]))
            UVy.append(abs(float(UVdata[1])))
    if UVy[0] > UVy[1] and UVy[0] > min_uv_threshold:
        UVmaxima.append(UVx[0])
    for i in range(1, len(UVy)-1):
        if UVy[i] > UVy[i-1] and UVy[i] > UVy[i+1] and UVy[i] > min_uv_threshold:
            UVmaxima.append(UVx[i])
    if UVy[-1] > UVy[-2] and UVy[-1] > min_uv_threshold:
        UVmaxima.append(UVx[-1])

        
    return UVmaxima

In [60]:
class rawData:
    def __init__(self, inputfile, row_no = 0, col_no = 0):
        #self.rawDADTable = pd.DataFrame(columns =['well', 'peakID', 'time', 'area', 'areaAbs', 'pStart', 'pEnd'])
        #self.rawUVTable = pd.DataFrame(columns =['well', 'peakID', 'time', 'UVvalue'])
        #self.rawMSTable = pd.DataFrame(columns =['well', 'peakID', 'time', 'MSvalue', 'MSintensity', "MStype"])
        #self.rawELSDTable = pd.DataFrame(columns =['well', 'peakID', 'time', 'area', 'areaAbs', 'pStart', 'pEnd'])
        self.row_no = row_no
        self.col_no = col_no
        
        with open(inputfile, errors = "ignore") as f:
            fullText = f.read()
            self.wellData = fullText.split("[SAMPLE]")[1:] #Split the file into individual wells
    
    def getWellFormat(self):
        #Get the plate dimensions from the first sample
        #Data sample: #Plate	01TL,XY,SD,1: 8,2:12,3: 90.0...
        #where number of rows is 8 and number of columns in 12
        #Overwrite the default option, but ensure that the user retains control
        #such that empty rows can be removed from the heatmap. 
        if self.row_no == 0 or self.col_no == 0:
            self.row_no = int(self.wellData[0].split("\n")[17].split(",")[3].split(":")[1])
            self.col_no = int(self.wellData[0].split("\n")[17].split(",")[4].split(":")[1])
            self.plate_cols_for_extraction = self.col_no
        else:
            self.plate_cols_for_extraction = int(self.wellData[0].split("\n")[17].split(",")[4].split(":")[1])
            
            
        #Find from rpt file how each well is specified 
        #Data sample: #Plate	01TL,XY,SD,1: 8,2:12,3: 90.0...
        self.row_col_order = self.wellData[0].split("\n")[17].split(",")[1]
        self.well_type = self.wellData[0].split("\n")[17].split(",")[2]
        
    def getWell(self, position):
        well_number = -1
        #If the well type is just a Single Digit...
        if self.well_type == "SD":
            #If the well is simply an integer between 1 and infinity
            #Single line function to trim full string to just the well number used
            well_number = int(self.wellData[position].split("Well")[1].split("\n")[0].split(":")[1].strip()) 
            #If the column number specified by the user is different to that found in the rpt file, 
            #this is the result of the user looking to trim off blank columns. Only wells described by 
            #a single digit need to be modified to take this into account. 
            if self.col_no != self.plate_cols_for_extraction:
                well_number = math.floor(well_number / self.plate_cols_for_extraction)*self.col_no + (well_number % self.plate_cols_for_extraction)

        #If the well type is a combination of letters/numbers...
        else:
            #Find if the well  
            if self.row_col_order == "XY":
                column = self.wellData[position].split("Well")[1].split("\n")[0].split(":")[1].split(",")[0].strip()
                row = self.wellData[position].split("Well")[1].split("\n")[0].split(":")[1].split(",")[1].strip()
            else: 
                column = self.wellData[position].split("Well")[1].split("\n")[0].split(":")[1].split(",")[1].strip()
                row = self.wellData[position].split("Well")[1].split("\n")[0].split(":")[1].split(",")[0].strip()

            #Convert the column to integer, either by direct
            #conversion, or by finding position in the alphabet
            try:
                col_as_int = int(column)
            except:
                col_as_int = ord(column.capitalize()) - 64

            #Convert the row to integer, either by direct
            #conversion, or by finding position in the alphabet
            try: 
                row_as_int = int(row)
            except: 
                row_as_int = ord(row.capitalize()) - 64

            #Calculate the wellno as a single integer
            well_number = (row_as_int - 1) * self.plate_cols_for_extraction + col_as_int
        return well_number
    
    def processDAD(self):
        peak_list = []
        for i in range(len(self.wellData)):
            functions = self.wellData[i].split("[FUNCTION]")
            well_number = self.getWell(i)
            for j in range(len(functions[1:])):
                function = functions[1:][j]
                lines = function.split("\n")
                
                #get peakarea for this peak
                if "Type\tDAD" in lines[4]:
                    chromatograms = function.split("[CHROMATOGRAM]")[1:]
                    for chromatogram in chromatograms:
                        c_lines = chromatogram.split("\n")
                        if "Description\tDAD:" in c_lines[3]:
                            spectra = function.split("[SPECTRUM]")[1:] #split by spectrum (i.e. each peak)
                            #chroma[wellno] = getChromatogram(chromatogram.split("[TRACE]")[1]) #get chromatogram for this well
                            peaks = chromatogram.split("[PEAK]")[1:]
                            for peak in peaks:
                                new_entry = {
                                    "well": well_number,
                                    "peakID": int(peak.split("Peak ID")[1].split("\n")[0].strip()),
                                    "time": float(peak.split("Time")[1].split("\n")[0].strip()),
                                    "pStart": peak.split("Peak\t")[1].split("\n")[0].split("\t")[0],
                                    "pEnd": peak.split("Peak\t")[1].split("\n")[0].split("\t")[1],
                                    "area": float(peak.split("Area %Total")[1].split("\n")[0].strip()),
                                    "areaAbs": float(peak.split("AreaAbs")[1].split("\n")[0].strip())
                                }
                                peak_list.append(new_entry)
        
        self.rawDADTable = pd.DataFrame(peak_list)
        
        
    def processUV(self, min_uv_threshold = 20): 
        peak_list = []
        for i in range(len(self.wellData)):
            functions = self.wellData[i].split("[FUNCTION]")
            well_number = self.getWell(i)
            for j in range(len(functions[1:])):
                function = functions[1:][j]
                lines = function.split("\n")
                if "Type\tDAD" in lines[4]:
                    spectra = function.split("[SPECTRUM]")[1:] #split by spectrum (i.e. each peak)

                    for spectrum in spectra:
                        UVdata = getUVData(spectrum, min_uv_threshold)
                        for maxima in UVdata:
                            new_entry = {
                                "well": well_number,
                                "peakID": int(spectrum.split("Peak ID")[1].split("\n")[0].strip()),
                                "time": float(spectrum.split("Time")[1].split("\n")[0].strip()),
                                "UVvalue": maxima,
                            }
                            peak_list.append(new_entry)

        
        self.rawUVTable = pd.DataFrame(peak_list)
        
        
    def processMS(self):
        peak_list = []
        for i in range(len(self.wellData)):
            functions = self.wellData[i].split("[FUNCTION]")
            well_number = self.getWell(i)
            for j in range(len(functions[1:])):
                function = functions[1:][j]
                lines = function.split("\n")
                if "IonMode\tES" in lines[3]:
                    
                    spectra = function.split("[SPECTRUM]")[1:] #split by spectrum (i.e. each peak)
                    for spectrum in spectra:
                        if "IonMode\tES+" in lines[3]:
                            MStype = "+"
                        elif "IonMode\tES-" in lines[3]:
                            MStype = "-"

                        MSdata = getMSData(spectrum)
                        for ion in MSdata:
                            new_entry = {
                                "well": well_number,
                                "peakID": int(spectrum.split("Peak ID")[1].split("\n")[0].strip()),
                                "time": float(spectrum.split("Time")[1].split("\n")[0].strip()),
                                "MSvalue": ion[0],
                                "MSintensity": ion[1],
                                "MStype": MStype
                            }
                            peak_list.append(new_entry)

        
        self.rawMSTable = pd.DataFrame(peak_list)
        
        #Calculate the sum of the MSintensities in each peak, then calculate the of each MSintensity to this total
        total_intensities = self.rawMSTable.groupby(["well", "peakID", "MStype"]).agg(total_intensity=('MSintensity', 'sum'))
        self.rawMSTable = self.rawMSTable.join(total_intensities, on=["well", "peakID", "MStype"], rsuffix = "right")
        self.rawMSTable["perc_intensity"] = 100 * self.rawMSTable["MSintensity"] / self.rawMSTable["total_intensity"]
        

            

In [61]:
inputfile = "example_dataset/Waters/Example2/LC-MS Data for 48-Well Plate.rpt"
#inputfile = "C:/Users/joe.mason/OneDrive - Domainex/Desktop/test.rpt"
test = rawData(inputfile)

In [62]:
test.getWellFormat()

In [63]:
test.processDAD()
test.processMS()
test.processUV()

In [64]:
test.rawMSTable

,well,peakID,time,MSvalue,MSintensity,MStype,total_intensity,perc_intensity
0,1,1,0.5019,114.09,8.840,+,218.289,4.049677
1,1,1,0.5019,121.13,16.400,+,218.289,7.512976
2,1,1,0.5019,195.16,100.000,+,218.289,45.810829
3,1,1,0.5019,196.28,11.140,+,218.289,5.103326
4,1,1,0.5019,212.20,4.815,+,218.289,2.205791
...,...,...,...,...,...,...,...,...
12852,48,9,1.3582,603.41,6.889,-,273.424,2.519530
12853,48,10,1.4865,212.19,100.000,-,110.840,90.220137
12854,48,10,1.4865,215.10,3.151,-,110.840,2.842837
12855,48,10,1.4865,232.91,3.711,-,110.840,3.348069


In [65]:
test.rawDADTable

,well,peakID,time,pStart,pEnd,area,areaAbs
0,1,1,0.5019,0.4911,0.5152,1.01,1.502801e+04
1,1,2,0.5240,0.5152,0.5636,1.29,1.927870e+04
2,1,3,0.7869,0.7736,0.8128,70.55,1.054612e+06
3,1,4,0.8886,0.8644,0.9128,1.57,2.346377e+04
4,1,5,1.1278,1.1086,1.1715,15.91,2.377554e+05
...,...,...,...,...,...,...,...
447,48,6,1.0382,1.0303,1.0536,2.08,7.462512e+04
448,48,7,1.1249,1.1086,1.1657,10.25,3.684584e+05
449,48,8,1.2911,1.2748,1.3119,2.45,8.824859e+04
450,48,9,1.3582,1.3428,1.3953,2.53,9.083438e+04


In [107]:
class Assignment:
    def __init__(self, filename, plate_col_no):   
        #read csv file into dataframe
        #replace empty cells with an empty string
        #convert all column names to lower case and remove whitespace
        self.inputCSV = pd.read_csv(filename)
        self.inputCSV.fillna("", inplace=True)
        self.inputCSV.columns = self.inputCSV.columns.str.strip().str.lower()
        
        self.plate_col_no = plate_col_no
        
        
    
    #Fn to convert a well name like B5 to machine format (11)
    def convertWellToNum(self, wells):
        result = []
        
        for well in wells:
            row = well[0]
            column = well[1:]
            #if the format of the row/column conforms to expectations
            if isinstance(int(column[0]), int):
                result.append(int((ord(row) - 65) * self.plate_col_no + int(column)))
            #Plates with more than 26 rows are unsupported at present. 
            else:
                logging.info("The well specified implies an unsupported plate.")
                sys.exit(2)
        return result
    
    #Fn to convert smiles into canonicalised smiles
    def getCanonSmiles(self, smiles):
        mol = Chem.MolFromSmiles(smiles.strip())
        return Chem.MolToSmiles(mol)
            
    def generateCPTable(self):
        compound_list = []
        type_dic = {
            "desired product smiles": "Product",
            "limiting reactant smiles":"Limiting Reactant",
            "internalstd smiles": "InternalSTD"
        }
        counter = {
            "desired product smiles": 1,
            "limiting reactant smiles": 1,
            "byproduct": 1
        }
        
        #Canonicalise all the incoming smiles in key columns in case the user hadn't done so already
        for col in self.inputCSV:
            if col in type_dic or ("byproduct" in col and "smiles" in col):
                self.inputCSV[col] = self.inputCSV[col].apply(self.getCanonSmiles)
        
        for col in self.inputCSV:
            if col in type_dic or ("byproduct" in col and "smiles" in col):
                cpname_column = f'{col.split(" smiles")[0]} name'
                cprt_column = f'{col.split(" smiles")[0]} rt'
                
                #group the input CSV by canonical smiles in that column, aggregated the wells into a list
                groupeddf = self.inputCSV.groupby(col, as_index=False)[["well"]].agg(lambda x: list(x))
                
                #Iterate through each of those grouped entries
                for index, row in groupeddf.iterrows():
                    cpindex = row[col]
                    cptype = type_dic[col] if col in type_dic else "byproduct"
                    #get a name for the compound if one was provided
                    name = ""
                    if cpname_column in self.inputCSV.columns:
                        name_series = self.inputCSV.loc[self.inputCSV[col] == row[col]][cpname_column]
                        potential_names = [x for x in name_series if x != ""]
                        if len(potential_names) != 0:
                            name = potential_names[0]
                    #if a name could not be generated, create a generic one using a simple counter
                    if name == "":
                        if col == "internalstd smiles":
                            name = "InternalSTD"
                        elif col in counter:
                            name = f'{type_dic[col]}{counter[col]}'
                            counter[col] = counter[col] + 1
                        elif "byproduct" in col:
                            name = f'Byproduct{counter["byproduct"]}'
                            counter["byproduct"] = counter["byproduct"] + 1
                    
                    #get a rentention time for the compound if one was provided
                    rt = 0
                    if cprt_column in self.inputCSV.columns:
                        rt_series = self.inputCSV.loc[self.inputCSV[col] == row[col]][cprt_column]
                        potential_rt = [x for x in rt_series if x != ""]
                        if len(potential_rt) != 0:
                            rt = potential_rt[0]
                            
                    new_entry = {
                        "smiles": cpindex,
                        "type": cptype,
                        "locations": row["well"],
                        "name": name,
                        "rt": rt,
                        "comments": []
                    }
                    compound_list.append(new_entry)
        
        self.cpTable = pd.DataFrame(compound_list)
        
        #set the index to be the canonicalised smiles
        self.cpTable.index = list(self.cpTable["smiles"])
        
        #convert the "A1" style well IDs into a integer, to allow matching to a well in the rawData
        self.cpTable["locations"] = self.cpTable["locations"].apply(self.convertWellToNum)

    
    def generateEMs(self, calc_boc):
        
        def getMW(smiles):
            mol = Chem.MolFromSmiles(smiles)
            return round(Descriptors.ExactMolWt(mol), 2)
        
        def transform_and_getMW(smiles, smirks, stage):
            try:
                mol = Chem.MolFromSmiles(smiles)
                rxn1 = AllChem.ReactionFromSmarts(smirks)
                new_mol1 = rxn1.RunReactants((mol, ))[0][0]
                #Sanitise the molecule to make sure that a sensible molecule was produced. 
                Chem.SanitizeMol(new_mol1)
                return round(Descriptors.ExactMolWt(new_mol1), 2)
            except:
                if ("Cl" in smiles or "Br" in smiles) and stage == "mass2":
                    mol = Chem.MolFromSmiles(smiles)
                    return round(Descriptors.ExactMolWt(mol), 2) + 2
                else:
                    return 0
            
        self.cpTable["mass1"] = self.cpTable["smiles"].apply(lambda smiles: getMW(smiles))
        
        if calc_boc == "True":
            
            smirks1 = "[NX3,n:1][C:2](=[O:3])[O:4][C]([CH3])([CH3])[CH3]>>[*:1][C:2](=[O:3])[O:4]"
            self.cpTable["mass2"] = self.cpTable["smiles"].apply(lambda smiles: transform_and_getMW(smiles, smirks1, "mass2"))
            
            smirks2 = "[NX3,n:1][C](=[O])[O][C]([CH3])([CH3])[CH3]>>[*:1][H]"
            self.cpTable["mass3"] = self.cpTable["smiles"].apply(lambda smiles: transform_and_getMW(smiles, smirks2, "mass3"))
            

    def findHits(self, rawData, detector, mass_abs_tol = 0.5, min_massconf_threshold = 10, 
                                      time_abs_tol = 0.025, calc_higherions = "True"):
        
        def getMatches(compound):
            if detector == "UV":
                chromadata = rawData.rawDADTable.loc[rawData.rawDADTable["well"].isin(compound["locations"])]
            else: 
                chromadata = rawData.rawELSDTable.loc[rawData.rawELSDTable["well"] in compound["locations"]]
            
            MSdata = rawData.rawMSTable.loc[rawData.rawMSTable["well"].isin(compound["locations"])]
            
            MS_hits = []
            
            for mass in [compound["mass1"], compound["mass2"], compound["mass3"]]:
                hits = MSdata.loc[(MSdata["MStype"] == "+") & 
                                   ((abs(MSdata["MSvalue"] - (mass + 1.01)) <= mass_abs_tol) |
                                    (abs(MSdata["MSvalue"] - (mass + 2.02)/2) <= mass_abs_tol) |
                                    (abs(MSdata["MSvalue"] - (mass + 3.03)/3) <= mass_abs_tol))]
                MS_hits = MS_hits + list(hits.index.values)
                
                hits = MSdata.loc[(MSdata["MStype"] == "-") & 
                                   (abs(MSdata["MSvalue"] - (mass - 1.01)) <= mass_abs_tol)]
                MS_hits = MS_hits + list(hits.index.values)
            
            grouped = MSdata[MSdata.index.isin(MS_hits)].groupby(["well", "peakID"], as_index = False).agg(mass_conf = ("perc_intensity", "sum"))
            
            grouped = grouped.loc[grouped["mass_conf"] >= min_massconf_threshold]

            return grouped.to_dict("records")                      

        self.cpTable["hits"] = self.cpTable.apply(getMatches, axis = 1)
    
    def validateHits(self, rawData, detector, time_abs_tol = 0.025):
        
        def getRelevantPeaks(x, data):
            
            return list(data[(data["well"] == x["well"]) & (data["peakID"] == x["peakID"])].index.values)
            
        def clusterHits(compound):
                
            relevant_indexes = []
            for hit in compound["hits"]:
                relevant_indexes = relevant_indexes + getRelevantPeaks(hit, df)
            
            df2 = df[df.index.isin(relevant_indexes)]
            df2.sort_values("time", inplace = True)
            
            clusters = []
            for index in df2.index:
                if len(clusters) == 0:
                    clusters.append([index])
                else:
                    clusterFound = False
                    for cluster in clusters:
                        mean_rt = mean([df2.loc[i, "time"] for i in cluster])

                        if abs(mean_rt - df2.loc[index, "time"]) < time_abs_tol:
                            cluster.append(index)
                            clusterFound = True
                            break
                    if not clusterFound:
                        clusters.append([index])
                        
            return clusters
        
        def getClusterBand(compound):
            clusterbands = []
            for cluster in compound["clusters"]:
                mean = df.loc[df.index.isin(cluster)]["time"].mean()
                clusterbands.append(round(mean, 5))
            return clusterbands
                       
        def selectCluster_ifrt(row):
            comments = row["comments"]
            #If the user has specified a retention time, we should select only the cluster 
            #that is closest to that retention time, and within the specified time_abs_tol
            if row["rt"] != 0:
                suitable_clusters = [index for index, i in enumerate(row["cluster_bands"]) if abs(i - row["rt"]) < time_abs_tol]

                #If there is more than one cluster close to the specified retention time
                #take only the cluster which is closest 
                if len(suitable_clusters) > 1:
                    diffs = [row["cluster_bands"][index]-row["rt"] for index in suitable_clusters]
                    index_min = min(range(len(diffs)), key=diffs.__getitem__)
                    row["clusters"] = [row["clusters"][suitable_clusters[index_min]]]
                    #Update the cluster bands to only include the correct label
                    cluster_bands = [row["cluster_bands"][suitable_clusters[index_min]]]

                    row["comments"].append("<strong>Multiple clusters were found close the specified"
                                        " retention time.</strong>")
                    row["comments"].append(f'<strong>Cluster {index_min} was selected as it was closest'
                                        ' to the specified retention time.</strong>')
                elif len(suitable_clusters) == 1:
                    row["clusters"] = [row["clusters"][suitable_clusters[0]]]
                    row["cluster_bands"] = [row["cluster_bands"][suitable_clusters[0]]]
                    row["comments"].append("<strong>A single cluster was found close the specified"
                                        " retention time and this was selected for analysis.</strong>")
                else:
                    row["comments"].append("<strong>No cluster was found near to the specified "
                                        "retention time. Proceeding with analysis using all "
                                        f'{len(row["clusters"])} clusters.</strong>')
            return row
            
        def refineClustersByTime(row):
            """
            Takes in input cluster of all the hit peaks, 
            and refines them by finding a mid-value for the retention
            time based on which hit has the greatest number of nearest neighbours. 
            Sorts the best hits into "green", uncertain ones into "orange" 
            and those where another peak closer to the mid-value was found
            in the same well into "discarded". 

            :param cluster: list of dictionaries, where each dictionary is a hit
            :param comments: A list of comments for that structure so far.

            :return: List comprising [a dictionary for the refined cluster, list of comments]
            """

            [clusters, comments, expected_rt] = [row["clusters"], row["comments"], row["rt"]]
            refined_clusters = []

            for cluster in clusters:
                refined_cluster = {
                    "green":[],
                    "orange": [],
                    "discarded": [],
                }
                list = [get rt, get the area, ]
                mid_values = []
                mid_value = 0

                if expected_rt != 0:
                    mid_value = expected_rt
                else: 
                    for i in cluster:
                        mid_value = df.loc[i, "time"]
                        mid_values.append([mid_value, len([df.loc[j, "time"] for j in cluster if abs(df.loc[i, "time"] - mid_value) < time_abs_tol/4)])])
                    mid_value = max(mid_values, key = lambda x: x[1])[0]

                #sort the peaks by the well they occupy
                peaks_by_wells = {}
                for i in cluster:
                    if df.loc[i, "well"] not in peaks_by_wells:
                        peaks_by_wells[df.loc[i, "well"]] = []
                    peaks_by_wells[df.loc[i, "well"]].append([df.loc[i, "well"]])

                #For each well, select the peak that's closest to the mid-value in cases
                #where there was more than one hit in that cluster in one well
                for index, well in peaks_by_wells.items():
                    if len(well) > 1:
                        min_diff = min([abs(x["time"]-mid_value) for x in well])

                    for peak in well:

                        if abs(peak["time"]-mid_value) == min_diff:
                            refined_cluster["green"].append(peak)
                        else:
                            refined_cluster["discarded"].append(peak)
                            comments.append(f'Peak at {peak["time"]} '
                                        f'in well {getUserReadableWell(peak["well"])} was discarded '
                                        'as there was an alternative peak '
                                        'in the same well which was closer to the '
                                        'mid-point of the cluster.')
                    else:
                        refined_cluster["green"].append(well[0])

                #Refine these hits further by finding those which are within time_abs_tol/2
                #of the mid-value. Any others are marked as tentative and the user is alerted.         
                ref2_cluster = {
                    "green":[],
                    "orange": refined_cluster["orange"],
                    "discarded": refined_cluster["discarded"],
                }        

                for peak in refined_cluster["green"]:
                    if math.isclose(peak["time"], mid_value, abs_tol = options.time_abs_tol / 2):
                        ref2_cluster["green"].append(peak)
                    else:
                        ref2_cluster["orange"].append(peak)
                        comments.append(f'Peak at {peak["time"]} in '
                                       f'well {getUserReadableWell(peak["well"])} was '
                                       'marked as tentative as it was found to be too '
                                       'far from the mid-value of the cluster.')
                refined_clusters.append(ref2_cluster)

            row["clusters"] = refined_clusters
            return row
        
        #Select the dataframe to be used for all functions above. 
        if detector == "UV":
            df = rawData.rawDADTable
        else:
            df = rawData.rawELSDTable
    
        self.cpTable["clusters"] = self.cpTable.apply(clusterHits, axis = 1)
        self.cpTable["cluster_bands"] = self.cpTable.apply(getClusterBand, axis = 1)
        self.cpTable = self.cpTable.apply(selectCluster_ifrt, axis = 1)
        
        self.cpTable = self.cpTable.apply(refineClustersByTime, axis = 1)
        
        
        
        
        
            
        
        
            
            
                
            
            
        
        
    

In [108]:
cpTable = Assignment("example_dataset/Waters/Example2/PyParse_designer_platemap.csv", 12)

C:\Users\joe.mason\AppData\Local\Temp\ipykernel_22884\1881550284.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  self.inputCSV.fillna("", inplace=True)


In [109]:
cpTable.generateCPTable()

In [110]:
cpTable.generateEMs("True")

In [111]:
cpTable.findHits(test, "UV")

In [112]:
cpTable.cpTable

,smiles,type,locations,name,rt,comments,mass1,mass2,mass3,hits
OCc1coc(-c2cnc3ccccc3c2)n1,OCc1coc(-c2cnc3ccccc3c2)n1,Product,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",Product1,0,[],226.07,0.00,0,"[{'well': 2, 'peakID': 4, 'mass_conf': 35.7957..."
Brc1cnc2ccccc2c1,Brc1cnc2ccccc2c1,Limiting Reactant,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",Limiting Reactant1,0,[],206.97,208.97,0,"[{'well': 9, 'peakID': 9, 'mass_conf': 14.9063..."
c1ccc(CN(Cc2ccccc2)c2ccccc2)cc1,c1ccc(CN(Cc2ccccc2)c2ccccc2)cc1,InternalSTD,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",Dibenzylaniline,0,[],273.15,0.00,0,"[{'well': 1, 'peakID': 6, 'mass_conf': 79.5253..."
Oc1cnc2ccccc2c1,Oc1cnc2ccccc2c1,byproduct,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",phenol bp,0,[],145.05,0.00,0,"[{'well': 2, 'peakID': 3, 'mass_conf': 90.9605..."


In [113]:
cpTable.validateHits(test, "UV")

TypeError: 'int' object is not subscriptable

In [902]:
output = []

for i in result:
    thing = []
    for j in i:
        thing.append(test.rawDADTable.loc[j, "time"])
    output.append(thing)

In [903]:
output

[[0.5844, 0.5848, 0.5848, 0.5857],
 [0.6594],
 [0.7098,
  0.7103,
  0.7103,
  0.7103,
  0.7103,
  0.7103,
  0.7103,
  0.7103,
  0.7107,
  0.7107,
  0.7107,
  0.7107,
  0.7107,
  0.7107,
  0.7107,
  0.7107,
  0.7107,
  0.7111,
  0.7111,
  0.7111,
  0.7111,
  0.7111,
  0.7111,
  0.7111,
  0.7115,
  0.7115,
  0.7132,
  0.7178,
  0.7328,
  0.7328,
  0.7328,
  0.7332,
  0.7332,
  0.7332,
  0.7336,
  0.7336,
  0.7336,
  0.7336,
  0.7336,
  0.7336,
  0.734,
  0.734,
  0.734,
  0.734,
  0.734,
  0.734,
  0.734,
  0.734,
  0.734,
  0.734,
  0.734,
  0.734,
  0.734,
  0.734,
  0.734,
  0.734,
  0.734,
  0.734,
  0.734,
  0.734,
  0.7344,
  0.7344,
  0.7344,
  0.7348,
  0.7348],
 [0.7861,
  0.7861,
  0.7861,
  0.7861,
  0.7861,
  0.7865,
  0.7865,
  0.7865,
  0.7865,
  0.7869,
  0.7869,
  0.7869,
  0.7869,
  0.8036,
  0.8036],
 [0.8215, 0.8215, 0.8215, 0.8215, 0.8215, 0.8219, 0.8219, 0.8224]]

In [821]:
test.rawMSTable.loc[(test.rawMSTable["well"] == 48) & (test.rawMSTable["peakID"] == 10)]

,well,peakID,time,MSvalue,MSintensity,MStype,total_intensity,perc_intensity
12639,48,10,1.4865,274.28,100.000,+,176.807,56.558847
12640,48,10,1.4865,275.26,22.206,+,176.807,12.559457
12641,48,10,1.4865,469.49,34.215,+,176.807,19.351609
12642,48,10,1.4865,470.49,14.038,+,176.807,7.939731
12643,48,10,1.4865,646.41,6.348,+,176.807,3.590356
12853,48,10,1.4865,212.19,100.000,-,110.840,90.220137
12854,48,10,1.4865,215.10,3.151,-,110.840,2.842837
12855,48,10,1.4865,232.91,3.711,-,110.840,3.348069
12856,48,10,1.4865,236.34,3.978,-,110.840,3.588957


In [739]:
cpTable.cpTable


,smiles,type,locations,name,rt,mass1,mass2,mass3,hits
OCc1coc(-c2cnc3ccccc3c2)n1,OCc1coc(-c2cnc3ccccc3c2)n1,Product,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",Product1,0,226.07,0.00,0,"[{'well': 2, 'peakID': 4, 'mass_conf': 35.7957..."
Brc1cnc2ccccc2c1,Brc1cnc2ccccc2c1,Limiting Reactant,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",Limiting Reactant1,0,206.97,207.97,0,"[{'well': 9, 'peakID': 9, 'mass_conf': 10.9029..."
c1ccc(CN(Cc2ccccc2)c2ccccc2)cc1,c1ccc(CN(Cc2ccccc2)c2ccccc2)cc1,InternalSTD,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",Dibenzylaniline,0,273.15,0.00,0,"[{'well': 1, 'peakID': 6, 'mass_conf': 79.5253..."
Oc1cnc2ccccc2c1,Oc1cnc2ccccc2c1,byproduct,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",phenol bp,0,145.05,0.00,0,"[{'well': 2, 'peakID': 3, 'mass_conf': 84.7376..."


In [869]:
test.rawDADTable.loc[1, "time"]

0.524